In [7]:
import pandas as pd
from scipy import stats

# 1. Load both result files
df_nyt = pd.read_csv("../data/bot5_chronological_results.csv")
df_rew = pd.read_csv("../data/bot5_chronological_results_2.csv")

# 2. Keep only the FIRST instance of each target word
df_nyt_dedup = df_nyt.drop_duplicates(subset=['target'], keep='first').copy()
df_rew_dedup = df_rew.drop_duplicates(subset=['target'], keep='first').copy()

# 3. Merge deduplicated datasets
merged = df_nyt_dedup.merge(df_rew_dedup, on="target", suffixes=("_nyt", "_rew"))

def analyze_significance(df, label):
    diffs = df["guess_count_nyt"] - df["guess_count_rew"]
    non_zero_diffs = diffs[diffs != 0]
    
    if len(non_zero_diffs) > 0:
        t_stat, p_val_ttest = stats.ttest_rel(df["guess_count_nyt"], df["guess_count_rew"])
        stat_wilc, p_val_wilc = stats.wilcoxon(non_zero_diffs)
    else:
        p_val_ttest, p_val_wilc = 1.0, 1.0

    print("=" * 60)
    print(f"--- STATISTICAL SIGNIFICANCE: {label} ---")
    print("=" * 60)
    print(f"Total Unique Targets Evaluated: {len(df):,}")
    print(f"Net Guesses Saved:              {diffs.sum()}")
    print(f"Games Improved (+1+):           {(diffs > 0).sum()}")
    print(f"Games Worsened (-1+):           {(diffs < 0).sum()}")
    print(f"Games Identical:                {(diffs == 0).sum()}")
    print("-" * 60)
    print(f"Paired t-test p-value:          {p_val_ttest:.6f}")
    print(f"Wilcoxon p-value:               {p_val_wilc:.6f}")
    
    sig_status = "YES (Statistically Significant)" if p_val_ttest < 0.05 else "NO (Not Significant)"
    print(f"Significant at alpha = 0.05?    {sig_status}")
    print("=" * 60 + "\n")

# Run test on ALL unique target words
analyze_significance(merged, "ALL UNIQUE TARGETS")

# Run test on 2023 ONWARDS ONLY
merged_2023 = merged[merged["year_nyt"] >= 2023].copy()
analyze_significance(merged_2023, "2023 ONWARDS ONLY")

--- STATISTICAL SIGNIFICANCE: ALL UNIQUE TARGETS ---
Total Unique Targets Evaluated: 1,838
Net Guesses Saved:              40
Games Improved (+1+):           210
Games Worsened (-1+):           170
Games Identical:                1458
------------------------------------------------------------
Paired t-test p-value:          0.046554
Wilcoxon p-value:               0.046746
Significant at alpha = 0.05?    YES (Statistically Significant)

--- STATISTICAL SIGNIFICANCE: 2023 ONWARDS ONLY ---
Total Unique Targets Evaluated: 1,277
Net Guesses Saved:              37
Games Improved (+1+):           150
Games Worsened (-1+):           113
Games Identical:                1014
------------------------------------------------------------
Paired t-test p-value:          0.028903
Wilcoxon p-value:               0.029206
Significant at alpha = 0.05?    YES (Statistically Significant)



In [8]:
from pathlib import Path
import pandas as pd

# Load result CSVs
data_dir = Path("../data")
df_b5 = pd.read_csv(data_dir / "bot5_chronological_results_2.csv")
df_b6 = pd.read_csv(data_dir / "bot6_chronological_results.csv")

# Align Bot 5 with Bot 6's evaluation window using game_num
merged = pd.merge(
    df_b6,
    df_b5[["game_num", "guess_count"]],
    on="game_num",
    suffixes=("_b6", "_b5"),
)

# Subset masks
is_rep = merged["is_repeat"] == True
is_non_rep = merged["is_repeat"] == False

# Calculate metrics breakdown
summary_data = [
    {
        "Category": "Overall Era",
        "Games": len(merged),
        "Bot 5 Avg": merged["guess_count_b5"].mean(),
        "Bot 6 Avg": merged["guess_count_b6"].mean(),
    },
    {
        "Category": "Repeats Only",
        "Games": is_rep.sum(),
        "Bot 5 Avg": merged.loc[is_rep, "guess_count_b5"].mean(),
        "Bot 6 Avg": merged.loc[is_rep, "guess_count_b6"].mean(),
    },
    {
        "Category": "Non-Repeats Only",
        "Games": is_non_rep.sum(),
        "Bot 5 Avg": merged.loc[is_non_rep, "guess_count_b5"].mean(),
        "Bot 6 Avg": merged.loc[is_non_rep, "guess_count_b6"].mean(),
    },
]

# Print Plaintext Table
print(f"{'Category':<18} | {'Games':<6} | {'Bot 5 Avg':<10} | {'Bot 6 Avg':<10} | {'Diff (B6 - B5)':<14}")
print("-" * 70)

for row in summary_data:
    cat = row["Category"]
    games = row["Games"]
    b5_avg = row["Bot 5 Avg"]
    b6_avg = row["Bot 6 Avg"]
    diff = b6_avg - b5_avg
    print(f"{cat:<18} | {games:<6} | {b5_avg:<10.4f} | {b6_avg:<10.4f} | {diff:<+14.4f}")

Category           | Games  | Bot 5 Avg  | Bot 6 Avg  | Diff (B6 - B5)
----------------------------------------------------------------------
Overall Era        | 179    | 3.2793     | 3.1844     | -0.0950       
Repeats Only       | 19     | 4.6842     | 3.9474     | -0.7368       
Non-Repeats Only   | 160    | 3.1125     | 3.0938     | -0.0187       


In [9]:
from pathlib import Path
import pandas as pd

# 1. Load result CSVs for Bots 5, 6, and 7
data_dir = Path("../data")
df_b5 = pd.read_csv(data_dir / "bot5_chronological_results_2.csv")
df_b6 = pd.read_csv(data_dir / "bot6_chronological_results.csv")
df_b7 = pd.read_csv(data_dir / "bot7_chronological_results.csv")

# 2. Align Bot 5 and Bot 7 with Bot 6's evaluation window using game_num
merged = pd.merge(
    df_b6,
    df_b5[["game_num", "guess_count"]],
    on="game_num",
    suffixes=("_b6", "_b5"),
)
merged = pd.merge(
    merged,
    df_b7[["game_num", "guess_count"]].rename(columns={"guess_count": "guess_count_b7"}),
    on="game_num",
)

# 3. Subset masks
is_rep = merged["is_repeat"] == True
is_non_rep = merged["is_repeat"] == False

# 4. Calculate metrics breakdown across all three bots
summary_data = [
    {
        "Category": "Overall Era",
        "Games": len(merged),
        "Bot 5 Avg": merged["guess_count_b5"].mean(),
        "Bot 6 Avg": merged["guess_count_b6"].mean(),
        "Bot 7 Avg": merged["guess_count_b7"].mean(),
    },
    {
        "Category": "Repeats Only",
        "Games": is_rep.sum(),
        "Bot 5 Avg": merged.loc[is_rep, "guess_count_b5"].mean(),
        "Bot 6 Avg": merged.loc[is_rep, "guess_count_b6"].mean(),
        "Bot 7 Avg": merged.loc[is_rep, "guess_count_b7"].mean(),
    },
    {
        "Category": "Non-Repeats Only",
        "Games": is_non_rep.sum(),
        "Bot 5 Avg": merged.loc[is_non_rep, "guess_count_b5"].mean(),
        "Bot 6 Avg": merged.loc[is_non_rep, "guess_count_b6"].mean(),
        "Bot 7 Avg": merged.loc[is_non_rep, "guess_count_b7"].mean(),
    },
]

# 5. Print Plaintext Table
header_str = (
    f"{'Category':<18} | {'Games':<6} | {'Bot 5 Avg':<10} | "
    f"{'Bot 6 Avg':<10} | {'Bot 7 Avg':<10} | {'Diff (B6-B5)':<13} | {'Diff (B7-B5)':<13}"
)
print(header_str)
print("-" * len(header_str))

for row in summary_data:
    cat = row["Category"]
    games = row["Games"]
    b5_avg = row["Bot 5 Avg"]
    b6_avg = row["Bot 6 Avg"]
    b7_avg = row["Bot 7 Avg"]
    
    diff_b6_b5 = b6_avg - b5_avg
    diff_b7_b5 = b7_avg - b5_avg
    
    print(
        f"{cat:<18} | {games:<6} | {b5_avg:<10.4f} | "
        f"{b6_avg:<10.4f} | {b7_avg:<10.4f} | {diff_b6_b5:<+13.4f} | {diff_b7_b5:<+13.4f}"
    )

Category           | Games  | Bot 5 Avg  | Bot 6 Avg  | Bot 7 Avg  | Diff (B6-B5)  | Diff (B7-B5) 
--------------------------------------------------------------------------------------------------
Overall Era        | 179    | 3.2793     | 3.1844     | 3.1788     | -0.0950       | -0.1006      
Repeats Only       | 19     | 4.6842     | 3.9474     | 3.8947     | -0.7368       | -0.7895      
Non-Repeats Only   | 160    | 3.1125     | 3.0938     | 3.0938     | -0.0187       | -0.0187      


In [10]:
# 1. Load Bot 8 results
df_b8 = pd.read_csv(data_dir / "bot8_chronological_results.csv")

# 2. Merge Bot 8 into the existing aligned evaluation window
merged_678 = pd.merge(
    merged,
    df_b8[["game_num", "guess_count"]].rename(columns={"guess_count": "guess_count_b8"}),
    on="game_num",
)

# 3. Subset masks
is_rep_678 = merged_678["is_repeat"] == True
is_non_rep_678 = merged_678["is_repeat"] == False

# 4. Calculate metrics breakdown across Bots 6, 7, and 8
summary_data_678 = [
    {
        "Category": "Overall Era",
        "Games": len(merged_678),
        "Bot 6 Avg": merged_678["guess_count_b6"].mean(),
        "Bot 7 Avg": merged_678["guess_count_b7"].mean(),
        "Bot 8 Avg": merged_678["guess_count_b8"].mean(),
    },
    {
        "Category": "Repeats Only",
        "Games": is_rep_678.sum(),
        "Bot 6 Avg": merged_678.loc[is_rep_678, "guess_count_b6"].mean(),
        "Bot 7 Avg": merged_678.loc[is_rep_678, "guess_count_b7"].mean(),
        "Bot 8 Avg": merged_678.loc[is_rep_678, "guess_count_b8"].mean(),
    },
    {
        "Category": "Non-Repeats Only",
        "Games": is_non_rep_678.sum(),
        "Bot 6 Avg": merged_678.loc[is_non_rep_678, "guess_count_b6"].mean(),
        "Bot 7 Avg": merged_678.loc[is_non_rep_678, "guess_count_b7"].mean(),
        "Bot 8 Avg": merged_678.loc[is_non_rep_678, "guess_count_b8"].mean(),
    },
]

# 5. Print Plaintext Table
header_str_678 = (
    f"{'Category':<18} | {'Games':<6} | {'Bot 6 Avg':<10} | "
    f"{'Bot 7 Avg':<10} | {'Bot 8 Avg':<10} | {'Diff (B7-B6)':<13} | {'Diff (B8-B6)':<13}"
)
print(header_str_678)
print("-" * len(header_str_678))

for row in summary_data_678:
    cat = row["Category"]
    games = row["Games"]
    b6_avg = row["Bot 6 Avg"]
    b7_avg = row["Bot 7 Avg"]
    b8_avg = row["Bot 8 Avg"]
    
    diff_b7_b6 = b7_avg - b6_avg
    diff_b8_b6 = b8_avg - b6_avg
    
    print(
        f"{cat:<18} | {games:<6} | {b6_avg:<10.4f} | "
        f"{b7_avg:<10.4f} | {b8_avg:<10.4f} | {diff_b7_b6:<+13.4f} | {diff_b8_b6:<+13.4f}"
    )

Category           | Games  | Bot 6 Avg  | Bot 7 Avg  | Bot 8 Avg  | Diff (B7-B6)  | Diff (B8-B6) 
--------------------------------------------------------------------------------------------------
Overall Era        | 179    | 3.1844     | 3.1788     | 3.1397     | -0.0056       | -0.0447      
Repeats Only       | 19     | 3.9474     | 3.8947     | 3.5263     | -0.0526       | -0.4211      
Non-Repeats Only   | 160    | 3.0938     | 3.0938     | 3.0938     | +0.0000       | +0.0000      
